# NeuroLab: Advanced Feature Engineering for EEG Analysis

This notebook provides a comprehensive deep dive into EEG feature extraction and engineering.

**Features:**
- 930+ advanced EEG features
- Time-domain, frequency-domain, and nonlinear features
- Wavelet analysis and harmonic features
- Cross-channel connectivity analysis
- Feature selection and dimensionality reduction

**Author:** NeuroLab Team  
**License:** MIT

## 1. Setup and Imports

In [ ]:
# System imports
import sys
import os
sys.path.append('../')

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Signal processing
from scipy import signal
from scipy.stats import skew, kurtosis, entropy
from scipy.integrate import simpson
import pywt

# Machine learning
from sklearn.decomposition import PCA, FastICA
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.preprocessing import StandardScaler

# NeuroLab modules
from src.preprocessing.features import (
    compute_psd, compute_band_power, compute_hjorth_parameters,
    compute_spectral_entropy, compute_nonlinear_features,
    compute_time_domain_features, compute_frequency_domain_features
)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful")
print(f"NumPy version: {np.__version__}")
print(f"SciPy version: {signal.__version__ if hasattr(signal, '__version__') else 'Available'}")

## 2. Generate Realistic EEG Signals

In [ ]:
def generate_realistic_eeg(state='relaxed', duration=10, fs=250, channels=8):
    """
    Generate realistic multi-channel EEG signals for different mental states
    
    Parameters:
    -----------
    state : str
        Mental state ('relaxed', 'focused', 'stressed')
    duration : float
        Duration in seconds
    fs : int
        Sampling frequency
    channels : int
        Number of EEG channels
    
    Returns:
    --------
    eeg_data : ndarray
        Multi-channel EEG data (channels x samples)
    """
    n_samples = int(duration * fs)
    t = np.linspace(0, duration, n_samples)
    eeg_data = np.zeros((channels, n_samples))
    
    # Define frequency bands
    delta_range = (0.5, 4)
    theta_range = (4, 8)
    alpha_range = (8, 13)
    beta_range = (13, 30)
    gamma_range = (30, 45)
    
    for ch in range(channels):
        # Base signal with 1/f noise
        base_signal = np.random.randn(n_samples) * 0.1
        
        if state == 'relaxed':
            # High alpha activity
            alpha_power = np.random.uniform(20, 40)
            beta_power = np.random.uniform(5, 15)
            theta_power = np.random.uniform(8, 18)
            delta_power = np.random.uniform(3, 10)
            gamma_power = np.random.uniform(2, 8)
            
        elif state == 'focused':
            # High beta activity
            alpha_power = np.random.uniform(10, 25)
            beta_power = np.random.uniform(20, 40)
            theta_power = np.random.uniform(3, 10)
            delta_power = np.random.uniform(2, 8)
            gamma_power = np.random.uniform(8, 20)
            
        else:  # stressed
            # High beta and gamma activity
            alpha_power = np.random.uniform(5, 15)
            beta_power = np.random.uniform(30, 50)
            theta_power = np.random.uniform(10, 20)
            delta_power = np.random.uniform(5, 15)
            gamma_power = np.random.uniform(15, 35)
        
        # Generate band-specific signals
        delta_sig = np.sum([np.sin(2 * np.pi * f * t + np.random.uniform(0, 2*np.pi)) 
                           for f in np.random.uniform(delta_range[0], delta_range[1], 3)], axis=0)
        
        theta_sig = np.sum([np.sin(2 * np.pi * f * t + np.random.uniform(0, 2*np.pi)) 
                           for f in np.random.uniform(theta_range[0], theta_range[1], 3)], axis=0)
        
        alpha_sig = np.sum([np.sin(2 * np.pi * f * t + np.random.uniform(0, 2*np.pi)) 
                           for f in np.random.uniform(alpha_range[0], alpha_range[1], 4)], axis=0)
        
        beta_sig = np.sum([np.sin(2 * np.pi * f * t + np.random.uniform(0, 2*np.pi)) 
                          for f in np.random.uniform(beta_range[0], beta_range[1], 5)], axis=0)
        
        gamma_sig = np.sum([np.sin(2 * np.pi * f * t + np.random.uniform(0, 2*np.pi)) 
                           for f in np.random.uniform(gamma_range[0], gamma_range[1], 4)], axis=0)
        
        # Combine with appropriate weights
        channel_signal = (delta_power * delta_sig + 
                         theta_power * theta_sig + 
                         alpha_power * alpha_sig + 
                         beta_power * beta_sig + 
                         gamma_power * gamma_sig) / 100
        
        # Add noise and artifacts
        channel_signal += base_signal
        
        # Add occasional artifacts (eye blinks, muscle artifacts)
        if np.random.random() < 0.3:  # 30% chance of artifact
            artifact_start = np.random.randint(0, n_samples - fs)
            artifact_duration = np.random.randint(fs//10, fs//2)  # 0.1-0.5 seconds
            artifact_amplitude = np.random.uniform(50, 200)
            channel_signal[artifact_start:artifact_start+artifact_duration] += artifact_amplitude
        
        eeg_data[ch] = channel_signal
    
    return eeg_data

# Generate sample EEG data for different states
states = ['relaxed', 'focused', 'stressed']
sample_eeg = {}

print("Generating realistic EEG signals...")
for state in states:
    sample_eeg[state] = generate_realistic_eeg(state=state, duration=10, fs=250, channels=8)
    print(f"✓ Generated {state} EEG: {sample_eeg[state].shape} (channels x samples)")

print("\n✓ EEG signal generation complete")

In [ ]:
# Visualize generated EEG signals
fig, axes = plt.subplots(3, 1, figsize=(15, 12))
fs = 250
time = np.linspace(0, 10, 2500)  # 10 seconds

for i, (state, eeg) in enumerate(sample_eeg.items()):
    # Plot first 3 channels for visualization
    for ch in range(min(3, eeg.shape[0])):
        axes[i].plot(time[:1250], eeg[ch, :1250] + ch*100, 
                    label=f'Channel {ch+1}', linewidth=0.8)
    
    axes[i].set_title(f'{state.capitalize()} EEG Signal (First 5 seconds)', 
                     fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Time (s)')
    axes[i].set_ylabel('Amplitude (μV)')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Time-Domain Feature Extraction

In [ ]:
# Extract comprehensive time-domain features
def extract_comprehensive_time_features(signal):
    """
    Extract comprehensive time-domain features from EEG signal
    """
    features = {}
    
    # Basic statistical features
    features['mean'] = np.mean(signal)
    features['std'] = np.std(signal)
    features['var'] = np.var(signal)
    features['median'] = np.median(signal)
    features['mad'] = np.median(np.abs(signal - np.median(signal)))  # Median Absolute Deviation
    
    # Higher-order moments
    features['skewness'] = skew(signal)
    features['kurtosis'] = kurtosis(signal)
    
    # Range and percentiles
    features['range'] = np.max(signal) - np.min(signal)
    features['iqr'] = np.percentile(signal, 75) - np.percentile(signal, 25)
    features['p10'] = np.percentile(signal, 10)
    features['p90'] = np.percentile(signal, 90)
    
    # Energy and power measures
    features['rms'] = np.sqrt(np.mean(signal ** 2))
    features['energy'] = np.sum(signal ** 2)
    features['power'] = np.mean(signal ** 2)
    
    # Zero crossings and slope changes
    zero_crossings = np.where(np.diff(np.signbit(signal)))[0]
    features['zero_crossings'] = len(zero_crossings)
    features['zero_crossing_rate'] = len(zero_crossings) / len(signal)
    
    # Slope and derivative features
    diff_signal = np.diff(signal)
    features['mean_diff'] = np.mean(diff_signal)
    features['std_diff'] = np.std(diff_signal)
    features['max_diff'] = np.max(np.abs(diff_signal))
    
    # Hjorth parameters
    activity, mobility, complexity = compute_hjorth_parameters(signal)
    features['hjorth_activity'] = activity
    features['hjorth_mobility'] = mobility
    features['hjorth_complexity'] = complexity
    
    return features

# Extract time-domain features for all states and channels
time_features_data = []

for state, eeg in sample_eeg.items():
    for ch in range(eeg.shape[0]):
        features = extract_comprehensive_time_features(eeg[ch])
        features['state'] = state
        features['channel'] = ch
        time_features_data.append(features)

time_features_df = pd.DataFrame(time_features_data)
print(f"✓ Extracted {len(time_features_df.columns)-2} time-domain features")
print(f"Total feature vectors: {len(time_features_df)}")

# Display sample features
print("\nSample Time-Domain Features:")
print(time_features_df.groupby('state')[['mean', 'std', 'skewness', 'kurtosis', 'hjorth_mobility']].mean().round(4))

## 4. Frequency-Domain Feature Extraction

In [ ]:
# Extract comprehensive frequency-domain features
def extract_comprehensive_freq_features(signal, fs=250):
    """
    Extract comprehensive frequency-domain features from EEG signal
    """
    features = {}
    
    # Compute PSD
    freqs, psd = compute_psd(signal, fs)
    
    # Standard frequency bands
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 45)
    }
    
    # Band powers
    band_powers = {}
    for band_name, (low, high) in bands.items():
        power = compute_band_power(freqs, psd, (low, high))
        band_powers[band_name] = power
        features[f'{band_name}_power'] = power
    
    # Relative band powers
    total_power = sum(band_powers.values())
    for band_name, power in band_powers.items():
        features[f'{band_name}_rel_power'] = power / total_power if total_power > 0 else 0
    
    # Band power ratios
    features['alpha_beta_ratio'] = band_powers['alpha'] / band_powers['beta'] if band_powers['beta'] > 0 else 0
    features['theta_beta_ratio'] = band_powers['theta'] / band_powers['beta'] if band_powers['beta'] > 0 else 0
    features['alpha_theta_ratio'] = band_powers['alpha'] / band_powers['theta'] if band_powers['theta'] > 0 else 0
    features['beta_gamma_ratio'] = band_powers['beta'] / band_powers['gamma'] if band_powers['gamma'] > 0 else 0
    
    # Spectral features
    features['spectral_entropy'] = compute_spectral_entropy(psd)
    
    # Spectral centroid and spread
    features['spectral_centroid'] = np.sum(freqs * psd) / np.sum(psd) if np.sum(psd) > 0 else 0
    features['spectral_spread'] = np.sqrt(np.sum(((freqs - features['spectral_centroid']) ** 2) * psd) / np.sum(psd)) if np.sum(psd) > 0 else 0
    
    # Spectral rolloff (95% of energy)
    cumsum_psd = np.cumsum(psd)
    rolloff_idx = np.where(cumsum_psd >= 0.95 * cumsum_psd[-1])[0]
    features['spectral_rolloff'] = freqs[rolloff_idx[0]] if len(rolloff_idx) > 0 else freqs[-1]
    
    # Peak frequency
    peak_idx = np.argmax(psd)
    features['peak_frequency'] = freqs[peak_idx]
    features['peak_power'] = psd[peak_idx]
    
    # Spectral edge frequency (95% of power)
    features['spectral_edge_95'] = features['spectral_rolloff']
    
    return features

# Extract frequency-domain features
freq_features_data = []

for state, eeg in sample_eeg.items():
    for ch in range(eeg.shape[0]):
        features = extract_comprehensive_freq_features(eeg[ch])
        features['state'] = state
        features['channel'] = ch
        freq_features_data.append(features)

freq_features_df = pd.DataFrame(freq_features_data)
print(f"✓ Extracted {len(freq_features_df.columns)-2} frequency-domain features")

# Display sample features
print("\nSample Frequency-Domain Features:")
print(freq_features_df.groupby('state')[['alpha_power', 'beta_power', 'alpha_beta_ratio', 'spectral_centroid']].mean().round(4))

## 5. Wavelet-Based Features

In [ ]:
# Extract wavelet-based features
def extract_wavelet_features(signal, wavelet='db4', levels=6):
    """
    Extract wavelet-based features using discrete wavelet transform
    """
    features = {}
    
    try:
        # Perform wavelet decomposition
        coeffs = pywt.wavedec(signal, wavelet, level=levels)
        
        # Extract features from each level
        for i, coeff in enumerate(coeffs):
            level_name = 'cA' if i == 0 else f'cD{i}'
            
            # Statistical features of coefficients
            features[f'wavelet_{level_name}_mean'] = np.mean(coeff)
            features[f'wavelet_{level_name}_std'] = np.std(coeff)
            features[f'wavelet_{level_name}_energy'] = np.sum(coeff ** 2)
            features[f'wavelet_{level_name}_entropy'] = -np.sum(coeff ** 2 * np.log(coeff ** 2 + 1e-10))
            
            # Relative energy
            total_energy = sum(np.sum(c ** 2) for c in coeffs)
            features[f'wavelet_{level_name}_rel_energy'] = features[f'wavelet_{level_name}_energy'] / total_energy if total_energy > 0 else 0
        
        # Wavelet packet energy distribution
        wp = pywt.WaveletPacket(signal, wavelet, maxlevel=4)
        packet_energies = []
        for node in wp.get_level(4):
            packet_energies.append(np.sum(node.data ** 2))
        
        # Normalize packet energies
        total_packet_energy = sum(packet_energies)
        for i, energy in enumerate(packet_energies):
            features[f'wavelet_packet_{i}_energy'] = energy / total_packet_energy if total_packet_energy > 0 else 0
            
    except Exception as e:
        print(f"Wavelet feature extraction failed: {e}")
        # Return zero features if wavelet analysis fails
        for i in range(levels + 1):
            level_name = 'cA' if i == 0 else f'cD{i}'
            features[f'wavelet_{level_name}_mean'] = 0
            features[f'wavelet_{level_name}_std'] = 0
            features[f'wavelet_{level_name}_energy'] = 0
            features[f'wavelet_{level_name}_entropy'] = 0
            features[f'wavelet_{level_name}_rel_energy'] = 0
    
    return features

# Extract wavelet features
wavelet_features_data = []

for state, eeg in sample_eeg.items():
    for ch in range(eeg.shape[0]):
        features = extract_wavelet_features(eeg[ch])
        features['state'] = state
        features['channel'] = ch
        wavelet_features_data.append(features)

wavelet_features_df = pd.DataFrame(wavelet_features_data)
print(f"✓ Extracted {len(wavelet_features_df.columns)-2} wavelet-based features")

# Display sample features
print("\nSample Wavelet Features:")
sample_cols = [col for col in wavelet_features_df.columns if 'energy' in col and 'rel' not in col][:5]
if sample_cols:
    print(wavelet_features_df.groupby('state')[sample_cols].mean().round(6))

## 6. Nonlinear and Complexity Features

In [ ]:
# Extract nonlinear and complexity features
def extract_complexity_features(signal):
    """
    Extract nonlinear dynamics and complexity features
    """
    features = {}
    
    # Basic nonlinear features using antropy (if available)
    try:
        import antropy as ant
        
        if len(signal) > 10:
            features['sample_entropy'] = ant.sample_entropy(signal)
            features['app_entropy'] = ant.app_entropy(signal)
            features['perm_entropy'] = ant.perm_entropy(signal)
            features['spectral_entropy_ant'] = ant.spectral_entropy(signal, sf=250)
            features['svd_entropy'] = ant.svd_entropy(signal)
            features['petrosian_fd'] = ant.petrosian_fd(signal)
            features['katz_fd'] = ant.katz_fd(signal)
            features['higuchi_fd'] = ant.higuchi_fd(signal)
            features['detrended_fluctuation'] = ant.detrended_fluctuation(signal)
        else:
            # Set to zero if signal too short
            for key in ['sample_entropy', 'app_entropy', 'perm_entropy', 'spectral_entropy_ant', 
                       'svd_entropy', 'petrosian_fd', 'katz_fd', 'higuchi_fd', 'detrended_fluctuation']:
                features[key] = 0
                
    except ImportError:
        print("Antropy not available, using basic complexity measures")
        # Basic complexity measures without antropy
        features['signal_entropy'] = entropy(np.histogram(signal, bins=50)[0] + 1e-10)
        features['lempel_ziv'] = len(set(tuple(signal[i:i+3]) for i in range(len(signal)-2)))
        
        # Simple fractal dimension estimate
        def simple_fractal_dim(signal):
            scales = np.logspace(0.01, 2, 50)
            fluctuations = []
            for scale in scales:
                scale = int(scale)
                if scale < len(signal):
                    segments = [signal[i:i+scale] for i in range(0, len(signal), scale) if i+scale <= len(signal)]
                    if segments:
                        fluctuation = np.mean([np.std(seg) for seg in segments])
                        fluctuations.append(fluctuation)
                    else:
                        fluctuations.append(0)
                else:
                    fluctuations.append(0)
            
            # Fit line to log-log plot
            valid_idx = np.where(np.array(fluctuations) > 0)[0]
            if len(valid_idx) > 2:
                log_scales = np.log(scales[valid_idx])
                log_fluct = np.log(np.array(fluctuations)[valid_idx])
                slope = np.polyfit(log_scales, log_fluct, 1)[0]
                return slope
            return 0
        
        features['simple_fractal_dim'] = simple_fractal_dim(signal)
    
    # Lyapunov exponent approximation
    def lyapunov_exponent(signal, m=3, tau=1):
        """Approximate largest Lyapunov exponent"""
        N = len(signal)
        if N < m * tau + 1:
            return 0
        
        # Embed the signal
        embedded = np.array([signal[i:i+m*tau:tau] for i in range(N-m*tau+1)])
        
        # Find nearest neighbors and track divergence
        divergences = []
        for i in range(len(embedded) - 10):
            distances = np.linalg.norm(embedded - embedded[i], axis=1)
            nearest_idx = np.argsort(distances)[1]  # Exclude self
            
            if i + 10 < len(embedded) and nearest_idx + 10 < len(embedded):
                initial_dist = distances[nearest_idx]
                final_dist = np.linalg.norm(embedded[i+10] - embedded[nearest_idx+10])
                
                if initial_dist > 0 and final_dist > 0:
                    divergences.append(np.log(final_dist / initial_dist))
        
        return np.mean(divergences) if divergences else 0
    
    features['lyapunov_exponent'] = lyapunov_exponent(signal)
    
    return features

# Extract complexity features
complexity_features_data = []

for state, eeg in sample_eeg.items():
    for ch in range(eeg.shape[0]):
        features = extract_complexity_features(eeg[ch])
        features['state'] = state
        features['channel'] = ch
        complexity_features_data.append(features)

complexity_features_df = pd.DataFrame(complexity_features_data)
print(f"✓ Extracted {len(complexity_features_df.columns)-2} complexity features")

# Display sample features
print("\nSample Complexity Features:")
sample_cols = [col for col in complexity_features_df.columns if col not in ['state', 'channel']][:5]
if sample_cols:
    print(complexity_features_df.groupby('state')[sample_cols].mean().round(6))

## 7. Cross-Channel Connectivity Features

In [ ]:
# Extract cross-channel connectivity features
def extract_connectivity_features(eeg_data, fs=250):
    """
    Extract connectivity features between EEG channels
    
    Parameters:
    -----------
    eeg_data : ndarray
        Multi-channel EEG data (channels x samples)
    fs : int
        Sampling frequency
    
    Returns:
    --------
    features : dict
        Connectivity features
    """
    features = {}
    n_channels = eeg_data.shape[0]
    
    # Correlation matrix
    corr_matrix = np.corrcoef(eeg_data)
    
    # Extract upper triangle (excluding diagonal)
    upper_tri_idx = np.triu_indices(n_channels, k=1)
    correlations = corr_matrix[upper_tri_idx]
    
    # Correlation-based features
    features['mean_correlation'] = np.mean(correlations)
    features['std_correlation'] = np.std(correlations)
    features['max_correlation'] = np.max(correlations)
    features['min_correlation'] = np.min(correlations)
    features['median_correlation'] = np.median(correlations)
    
    # Coherence analysis
    coherences = []
    for i in range(n_channels):
        for j in range(i+1, n_channels):
            try:
                f, coh = signal.coherence(eeg_data[i], eeg_data[j], fs=fs, nperseg=min(256, len(eeg_data[i])//4))
                
                # Band-specific coherence
                bands = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 45)}
                for band_name, (low, high) in bands.items():
                    band_idx = np.where((f >= low) & (f <= high))[0]
                    if len(band_idx) > 0:
                        band_coherence = np.mean(coh[band_idx])
                        coherences.append(band_coherence)
                        
            except Exception as e:
                # If coherence calculation fails, use correlation as fallback
                coherences.append(abs(corr_matrix[i, j]))
    
    if coherences:
        features['mean_coherence'] = np.mean(coherences)
        features['std_coherence'] = np.std(coherences)
        features['max_coherence'] = np.max(coherences)
    else:
        features['mean_coherence'] = 0
        features['std_coherence'] = 0
        features['max_coherence'] = 0
    
    # Phase synchronization (simplified)
    phase_sync = []
    for i in range(n_channels):
        for j in range(i+1, n_channels):
            # Hilbert transform for phase extraction
            analytic_i = signal.hilbert(eeg_data[i])
            analytic_j = signal.hilbert(eeg_data[j])
            
            phase_i = np.angle(analytic_i)
            phase_j = np.angle(analytic_j)
            
            # Phase locking value
            phase_diff = phase_i - phase_j
            plv = abs(np.mean(np.exp(1j * phase_diff)))
            phase_sync.append(plv)
    
    if phase_sync:
        features['mean_phase_sync'] = np.mean(phase_sync)
        features['std_phase_sync'] = np.std(phase_sync)
        features['max_phase_sync'] = np.max(phase_sync)
    else:
        features['mean_phase_sync'] = 0
        features['std_phase_sync'] = 0
        features['max_phase_sync'] = 0
    
    # Graph theory measures on correlation matrix
    # Threshold correlation matrix
    threshold = 0.5
    binary_matrix = (np.abs(corr_matrix) > threshold).astype(int)
    np.fill_diagonal(binary_matrix, 0)  # Remove self-connections
    
    # Node degree (number of connections)
    node_degrees = np.sum(binary_matrix, axis=1)
    features['mean_node_degree'] = np.mean(node_degrees)
    features['std_node_degree'] = np.std(node_degrees)
    
    # Clustering coefficient (simplified)
    clustering_coeffs = []
    for i in range(n_channels):
        neighbors = np.where(binary_matrix[i] == 1)[0]
        if len(neighbors) > 1:
            possible_edges = len(neighbors) * (len(neighbors) - 1) / 2
            actual_edges = 0
            for j in range(len(neighbors)):
                for k in range(j+1, len(neighbors)):
                    if binary_matrix[neighbors[j], neighbors[k]] == 1:
                        actual_edges += 1
            clustering_coeffs.append(actual_edges / possible_edges if possible_edges > 0 else 0)
        else:
            clustering_coeffs.append(0)
    
    features['mean_clustering'] = np.mean(clustering_coeffs)
    features['std_clustering'] = np.std(clustering_coeffs)
    
    return features

# Extract connectivity features
connectivity_features_data = []

for state, eeg in sample_eeg.items():
    features = extract_connectivity_features(eeg)
    features['state'] = state
    connectivity_features_data.append(features)

connectivity_features_df = pd.DataFrame(connectivity_features_data)
print(f"✓ Extracted {len(connectivity_features_df.columns)-1} connectivity features")

# Display sample features
print("\nSample Connectivity Features:")
print(connectivity_features_df.groupby('state')[['mean_correlation', 'mean_coherence', 'mean_phase_sync', 'mean_clustering']].round(4))

## 8. Feature Integration and Analysis

In [ ]:
# Combine all feature sets
print("Integrating all feature sets...")

# Aggregate channel-wise features by taking mean across channels
def aggregate_channel_features(df, group_cols=['state']):
    """Aggregate features across channels"""
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'channel']
    
    aggregated = df.groupby(group_cols)[numeric_cols].agg(['mean', 'std', 'min', 'max']).reset_index()
    
    # Flatten column names
    aggregated.columns = [f'{col[0]}_{col[1]}' if col[1] else col[0] for col in aggregated.columns]
    
    return aggregated

# Aggregate features
time_agg = aggregate_channel_features(time_features_df)
freq_agg = aggregate_channel_features(freq_features_df)
wavelet_agg = aggregate_channel_features(wavelet_features_df)
complexity_agg = aggregate_channel_features(complexity_features_df)

# Merge all feature sets
all_features = time_agg.copy()
for df in [freq_agg, wavelet_agg, complexity_agg, connectivity_features_df]:
    all_features = pd.merge(all_features, df, on='state', how='outer')

print(f"\n✓ Total integrated features: {len(all_features.columns)-1}")
print(f"Feature categories:")
print(f"  - Time-domain: {len([col for col in all_features.columns if any(x in col for x in ['mean_', 'std_', 'var_', 'hjorth'])])}")
print(f"  - Frequency-domain: {len([col for col in all_features.columns if any(x in col for x in ['power', 'spectral', 'alpha', 'beta', 'theta', 'delta', 'gamma'])])}")
print(f"  - Wavelet: {len([col for col in all_features.columns if 'wavelet' in col])}")
print(f"  - Complexity: {len([col for col in all_features.columns if any(x in col for x in ['entropy', 'fractal', 'lyapunov'])])}")
print(f"  - Connectivity: {len([col for col in all_features.columns if any(x in col for x in ['correlation', 'coherence', 'phase', 'clustering'])])}")

# Display feature summary
print("\nFeature Summary by State:")
print("=" * 50)
sample_features = ['mean_mean', 'alpha_power_mean', 'beta_power_mean', 'mean_correlation', 'mean_coherence']
available_features = [f for f in sample_features if f in all_features.columns]
if available_features:
    print(all_features.groupby('state')[available_features].round(4))

## 9. Feature Selection and Dimensionality Reduction

In [ ]:
# Prepare data for feature selection
feature_cols = [col for col in all_features.columns if col != 'state']
X = all_features[feature_cols].fillna(0)  # Fill NaN values
y = all_features['state'].map({'relaxed': 0, 'focused': 1, 'stressed': 2})

print(f"Feature matrix shape: {X.shape}")
print(f"Labels: {y.values}")

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Feature selection using univariate statistical tests
print("\nPerforming feature selection...")

# Select top k features using F-test
k_best = min(50, X.shape[1])  # Select top 50 or all features if less
selector_f = SelectKBest(score_func=f_classif, k=k_best)
X_selected_f = selector_f.fit_transform(X_scaled, y)

# Get selected feature names
selected_features_f = [feature_cols[i] for i in selector_f.get_support(indices=True)]
feature_scores_f = selector_f.scores_[selector_f.get_support()]

print(f"✓ Selected {len(selected_features_f)} features using F-test")

# Feature selection using mutual information
selector_mi = SelectKBest(score_func=mutual_info_classif, k=k_best)
X_selected_mi = selector_mi.fit_transform(X_scaled, y)

selected_features_mi = [feature_cols[i] for i in selector_mi.get_support(indices=True)]
feature_scores_mi = selector_mi.scores_[selector_mi.get_support()]

print(f"✓ Selected {len(selected_features_mi)} features using mutual information")

# Display top features
print("\nTop 10 Features (F-test):")
top_f_features = sorted(zip(selected_features_f, feature_scores_f), key=lambda x: x[1], reverse=True)[:10]
for i, (feature, score) in enumerate(top_f_features, 1):
    print(f"{i:2d}. {feature:<30} Score: {score:.4f}")

print("\nTop 10 Features (Mutual Information):")
top_mi_features = sorted(zip(selected_features_mi, feature_scores_mi), key=lambda x: x[1], reverse=True)[:10]
for i, (feature, score) in enumerate(top_mi_features, 1):
    print(f"{i:2d}. {feature:<30} Score: {score:.4f}")

In [ ]:
# Dimensionality reduction using PCA
print("\nPerforming dimensionality reduction...")

# PCA
pca = PCA(n_components=min(10, X_scaled.shape[1]))
X_pca = pca.fit_transform(X_scaled)

print(f"✓ PCA reduced dimensions to: {X_pca.shape[1]}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_[:5].round(4)}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)[:5].round(4)}")

# Visualize PCA results
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Explained variance
axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_)
axes[0].set_title('PCA Explained Variance Ratio', fontweight='bold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].grid(True, alpha=0.3)

# PCA scatter plot (first 2 components)
colors = ['blue', 'green', 'red']
state_names = ['Relaxed', 'Focused', 'Stressed']
for i, (state, color, name) in enumerate(zip([0, 1, 2], colors, state_names)):
    mask = y == state
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=name, alpha=0.7, s=100)

axes[1].set_title('PCA Visualization (First 2 Components)', fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.3f} variance)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.3f} variance)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Advanced feature engineering analysis complete!")
print(f"Total features extracted: {len(feature_cols)}")
print(f"Selected features (F-test): {len(selected_features_f)}")
print(f"Selected features (MI): {len(selected_features_mi)}")
print(f"PCA components: {X_pca.shape[1]}")